# PyTorch: Multi Class Dataset Classification

In [ ]:
import torch
from torch import nn
from torch import optim
from torch.utils.data import DataLoader, TensorDataset
from torchmetrics import Accuracy
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import common.torch as ct

In [ ]:
ct.set_default_seed()
ct.set_default_optimizations()
device = ct.get_optimal_device()

In [ ]:
print(f"PyTorch: version {torch.__version__}")
print(f"PyTorch: {device.type.upper()} device")

In [ ]:
# Hyperparameters
BATCH_SIZE = 64
N_EPOCHS = 100
# Other parameters
N_SAMPLES = 1000
N_CLASSES = 4
N_FEATURES = 2
THRESHOLD = 0.6

## Prepare Datasets

In [ ]:
x_blob, y_blob = make_blobs(n_samples=N_SAMPLES,
                            n_features=N_FEATURES,
                            centers=N_CLASSES,
                            cluster_std=0.75,
                            random_state=0)

In [ ]:
x_blob = torch.from_numpy(x_blob).type(torch.float)
y_blob = torch.from_numpy(y_blob).type(torch.long)

In [ ]:
# Split data into train and test datasets
x_train, x_test, y_train, y_test = train_test_split(x_blob, y_blob,
                                                    test_size=0.2,
                                                    random_state=0)

In [ ]:
tr_dl = DataLoader(TensorDataset(x_train, y_train), batch_size=BATCH_SIZE, num_workers=2, shuffle=True)
ts_dl = DataLoader(TensorDataset(x_test, y_test), batch_size=BATCH_SIZE, num_workers=2)
len(tr_dl), len(ts_dl)

In [ ]:
xs, ys = next(iter(tr_dl))

In [ ]:
plt.figure(figsize=(10, 10))
plt.scatter(x_blob[:, 0], x_blob[:, 1], c=y_blob, cmap='RdYlBu');

## Define Model

In [ ]:
class BlobModel(nn.Module):
    def __init__(self,
                 in_features: int = N_FEATURES,
                 out_features: int = N_CLASSES,
                 hidden_units: int = 8):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, out_features)
        )

    def forward(self, x):
        y = self.layers(x)
        return y

In [ ]:
model = BlobModel().to(device)

## Train Model

In [ ]:
logs_dir, writer = ct.get_summary_writer("pt_multi_class_dataset", "blob_model", "100_epochs")
logs_dir

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), momentum=0.9, lr=0.1)
accuracy = Accuracy(task='multiclass', num_classes=N_CLASSES).to(device)

In [ ]:
ct.train(model=model,
         tr_dl=tr_dl,
         ts_dl=ts_dl,
         optimizer=optimizer,
         criterion=criterion,
         metric=accuracy,
         n_epochs=N_EPOCHS,
         writer=writer,
         device=device)

## Evaluate Model

In [17]:
model.eval()
with torch.inference_mode():
    y_logits = model(x_test.to(device))
    y_pred = torch.softmax(y_logits, dim=1).argmax(dim=1)

loss = criterion(y_logits, y_test.to(device))
print(f'Final Loss: {loss:.3f}')

Final Loss: 0.096
